# 🎙️ Servidor de Clonagem de Voz (grátis, com GPU do Colab)

Este notebook sobe um servidor de **Text-to-Speech com clonagem de voz** usando o
modelo aberto **Coqui XTTS v2** e expõe ele na internet através de um túnel **ngrok**,
para que o app Android "Voz Clone" consiga se conectar e gerar áudios.

### Passo a passo
1. Menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.
2. Crie uma conta grátis em https://dashboard.ngrok.com/signup e pegue seu **Authtoken** em
   https://dashboard.ngrok.com/get-started/your-authtoken
3. Cole o token na célula abaixo (`NGROK_AUTHTOKEN`).
4. Rode todas as células em ordem (▶️ em cada uma, de cima para baixo).
5. Ao final, você vai receber uma URL pública tipo `https://xxxx.ngrok-free.app`.
6. Cole essa URL no app Android, no campo **"Servidor de clonagem de voz"**.

⚠️ Enquanto este notebook estiver aberto e rodando, o servidor fica no ar. Se fechar
a aba ou o Colab desconectar, você precisa rodar tudo de novo (e a URL muda).


## 1. Instalar dependências

In [ ]:
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok
!pip install -q coqui-tts


## 2. Configurar seu token do ngrok

In [ ]:
NGROK_AUTHTOKEN = "COLOQUE_SEU_TOKEN_AQUI"  # @param {type:"string"}

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN
print("Token configurado.")


## 3. Código do servidor (FastAPI + XTTS v2)

In [ ]:
%%writefile server.py
import io
import os
import tempfile
import traceback

from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

MODEL_NAME = "tts_models/multilingual/multi-dataset/xtts_v2"

app = FastAPI(title="Voz Clone - Servidor TTS")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

_tts_model = None


def get_model():
    global _tts_model
    if _tts_model is None:
        import torch
        from TTS.api import TTS

        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Carregando modelo {MODEL_NAME} em {device}...")
        _tts_model = TTS(MODEL_NAME).to(device)
        print("Modelo carregado com sucesso.")
    return _tts_model


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/tts")
async def tts(
    text: str = Form(...),
    language: str = Form("pt"),
    voice: UploadFile = File(...),
):
    if not text.strip():
        raise HTTPException(status_code=400, detail="Campo \'text\' vazio")

    try:
        model = get_model()

        suffix = os.path.splitext(voice.filename or "voice.wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as ref_tmp:
            ref_tmp.write(await voice.read())
            ref_path = ref_tmp.name

        out_path = ref_path + "_out.wav"

        try:
            model.tts_to_file(
                text=text,
                speaker_wav=ref_path,
                language=language,
                file_path=out_path,
            )

            with open(out_path, "rb") as f:
                audio_bytes = f.read()

            return Response(content=audio_bytes, media_type="audio/wav")
        finally:
            for p in (ref_path, out_path):
                try:
                    os.remove(p)
                except OSError:
                    pass

    except HTTPException:
        raise
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


## 4. Aceitar os termos do modelo XTTS v2 (Coqui)

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"
print("Termos de uso do modelo aceitos automaticamente (COQUI_TOS_AGREED=1).")


## 5. Subir o servidor e abrir o túnel público (ngrok)

Depois de rodar esta célula, procure no final da saída a linha:

```
🌍 URL pública do servidor: https://xxxx.ngrok-free.app
```

Copie essa URL e cole no app Android.

In [ ]:
import subprocess
import time
from pyngrok import ngrok

# Mata qualquer túnel/servidor anterior
ngrok.kill()
subprocess.run(["pkill", "-f", "uvicorn"], check=False)
time.sleep(1)

# Sobe o servidor em background
proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
)

print("Aguardando o servidor iniciar...")
time.sleep(8)

public_url = ngrok.connect(8000, "http")
print("=" * 60)
print(f"🌍 URL pública do servidor: {public_url}")
print("=" * 60)
print("Cole essa URL no app Android (campo 'Servidor de clonagem de voz').")
print("Deixe esta célula/notebook rodando enquanto for usar o app.")


## 6. (Opcional) Testar o servidor por aqui mesmo

Suba um áudio de exemplo (`exemplo_voz.wav`, alguns segundos de alguém falando)
usando o ícone de pasta 📁 à esquerda, arraste o arquivo para a raiz do Colab,
depois rode a célula abaixo para testar a geração antes de usar no app.

In [ ]:
import requests

with open("exemplo_voz.wav", "rb") as f:
    resp = requests.post(
        "http://localhost:8000/tts",
        data={"text": "Olá! Este é um teste da minha voz clonada.", "language": "pt"},
        files={"voice": f},
    )

if resp.status_code == 200:
    with open("teste_resultado.wav", "wb") as out:
        out.write(resp.content)
    print("Áudio gerado: teste_resultado.wav")
    from IPython.display import Audio, display
    display(Audio("teste_resultado.wav"))
else:
    print("Erro:", resp.status_code, resp.text)
